# De CSV depurado a Parquet

El notebook de hurtos deja dos tablas limpias en `data/processed/`:

- `hurtos_medellin_hechos.csv` — un registro por evento (~11 611 filas)
- `hurtos_medellin_bienes.csv` — un registro por bien reportado (~16 486 filas)

Aquí **no se vuelve a imputar ni a depurar**. Se toma ese CSV, se declaran otra vez los tipos (el CSV no los recuerda) y se escribe **Parquet**, el formato columnar que usarán pandas o Spark después.

La teoría está en [`bases de datos columnares.md`](../bases%20de%20datos%20columnares.md): filas vs columnas, qué es Parquet y una introducción a Spark.

In [ ]:
from pathlib import Path
import time

import pandas as pd

raiz = Path.cwd()
if not (raiz / "data" / "processed").exists():
    raiz = raiz.parent

procesados = raiz / "data" / "processed"
assert procesados.exists(), f"No está data/processed en {raiz}"
print("carpeta:", procesados.resolve())

## 1. Leer el CSV procesado

El CSV es texto: al reabrirlo, `fecha` vuelve a `object` y `coord_imputada` no es un contrato estable. Por eso el parseo se **repite aquí**, una última vez, antes de fosilizarlo en Parquet.

In [ ]:
hechos_csv = pd.read_csv(procesados / "hurtos_medellin_hechos.csv")
bienes_csv = pd.read_csv(procesados / "hurtos_medellin_bienes.csv")

print("hechos", hechos_csv.shape)
print("bienes", bienes_csv.shape)
print("\ndtypes al salir del CSV (casi todo es texto):")
print(hechos_csv.dtypes.value_counts().to_string())
hechos_csv.head(3)

## 2. Declarar tipos

Parquet guarda el esquema. Si se escribe con `fecha` como string, se pierde la ventaja. Las categóricas (`sexo`, `modalidad`, `comuna`…) se comprimen con diccionario: cada clase una vez, un entero corto por fila.

In [ ]:
DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
FRANJAS = ["Madrugada (0-5)", "Mañana (6-11)", "Tarde (12-17)", "Noche (18-23)"]
CATEGORICAS = [
    "sexo", "estado_civil", "medio_transporte", "modalidad", "conducta_especial",
    "arma_medio", "nombre_barrio", "codigo_barrio", "lugar", "sede_receptora",
    "bien", "categoria_bien", "color", "comuna",
]


def tipar(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["fecha"] = pd.to_datetime(out["fecha"], errors="coerce")
    out["anio_mes"] = pd.to_datetime(out["anio_mes"], errors="coerce")
    out["edad"] = pd.to_numeric(out["edad"], errors="coerce")
    out["lat"] = pd.to_numeric(out["lat"], errors="coerce")
    out["lon"] = pd.to_numeric(out["lon"], errors="coerce")
    for col in ("anio", "mes", "hora"):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")
    out["coord_imputada"] = out["coord_imputada"].astype(bool)
    out["edad_imputada"] = out["edad_imputada"].astype(bool)
    out["dia"] = pd.Categorical(out["dia"], categories=DIAS, ordered=True)
    out["franja"] = pd.Categorical(out["franja"], categories=FRANJAS, ordered=True)
    for col in CATEGORICAS:
        if col in out.columns:
            out[col] = out[col].astype("category")
    return out


hechos = tipar(hechos_csv)
bienes = tipar(bienes_csv)
print(hechos.dtypes.to_string())

## 3. Escribir Parquet

`engine="pyarrow"` es el motor estándar. `index=False` evita una columna extra `Unnamed: 0`. `compression="snappy"` es el default razonable: comprime bien y se descomprime rápido (mejor que `gzip` para análisis iterativo).

In [ ]:
ruta_hechos = procesados / "hurtos_medellin_hechos.parquet"
ruta_bienes = procesados / "hurtos_medellin_bienes.parquet"

hechos.to_parquet(ruta_hechos, engine="pyarrow", index=False, compression="snappy")
bienes.to_parquet(ruta_bienes, engine="pyarrow", index=False, compression="snappy")

print("escrito:")
print(" ", ruta_hechos.resolve())
print(" ", ruta_bienes.resolve())

## 4. CSV vs Parquet en disco

Menos bytes = menos I/O. La ganancia viene del diccionario en categóricas y de no repetir fechas y comas como texto.

In [ ]:
def mb(path: Path) -> float:
    return path.stat().st_size / (1024 * 1024)


comparacion = pd.DataFrame(
    {
        "csv_MB": [
            mb(procesados / "hurtos_medellin_hechos.csv"),
            mb(procesados / "hurtos_medellin_bienes.csv"),
        ],
        "parquet_MB": [mb(ruta_hechos), mb(ruta_bienes)],
    },
    index=["hechos", "bienes"],
)
comparacion["ratio_parquet/csv"] = comparacion["parquet_MB"] / comparacion["csv_MB"]
comparacion.round(3)

## 5. Proyección: leer solo las columnas del análisis

Un descriptivo o un modelo de modalidad no necesita `color`, `bien` ni la latitud textual. En CSV hay que parsear el archivo; en Parquet se piden las columnas y el resto no sale del disco. Eso es *projection pushdown*.

Con ~11 000 filas el cronómetro puede favorecer al CSV: abrir Parquet tiene un costo fijo. La ganancia aparece cuando la tabla es más ancha, más larga, o se lee muchas veces. Lo que ya se gana aquí es el **tamaño en disco** (sección 4) y **no tener que volver a parsear tipos**.

In [ ]:
columnas = ["hora", "comuna", "modalidad", "medio_transporte"]

t0 = time.perf_counter()
desde_csv = pd.read_csv(procesados / "hurtos_medellin_hechos.csv", usecols=columnas)
t_csv = time.perf_counter() - t0

t0 = time.perf_counter()
desde_parquet = pd.read_parquet(ruta_hechos, columns=columnas, engine="pyarrow")
t_parq = time.perf_counter() - t0

print(f"CSV, 4 columnas:     {t_csv:.4f} s  shape={desde_csv.shape}")
print(f"Parquet, 4 columnas: {t_parq:.4f} s  shape={desde_parquet.shape}")
print("tipos al salir de Parquet (ya no hay que volver a parsear):")
print(pd.read_parquet(ruta_hechos, engine="pyarrow").dtypes.to_string())

## 6. Round-trip

Misma cantidad de hechos y los mismos `id_hecho`. Si esto falla, no se sigue al modelo: el Parquet no es un derivado fiel.

In [ ]:
hechos_pq = pd.read_parquet(ruta_hechos, engine="pyarrow")
bienes_pq = pd.read_parquet(ruta_bienes, engine="pyarrow")

assert len(hechos_pq) == len(hechos), "filas de hechos no coinciden"
assert len(bienes_pq) == len(bienes), "filas de bienes no coinciden"
assert hechos_pq["id_hecho"].equals(hechos["id_hecho"].astype(hechos_pq["id_hecho"].dtype))

print("hechos", hechos_pq.shape, "| bienes", bienes_pq.shape)
print("id_hecho únicos (hechos):", hechos_pq["id_hecho"].nunique())
print("round-trip OK")

## 7. Spark, en breve

pandas carga la tabla en la RAM de **una** máquina. **Apache Spark** reparte filas y operaciones en un clúster (o en varios núcleos locales) y evalúa **en perezoso**: `select` / `where` / `groupBy` no corren hasta una acción (`count`, `write`).

Parquet es el formato nativo de ese motor: el footer trae esquema y min/max por bloque, así Spark no lee `color` si nadie lo pidió ni abre un *row group* que no puede cumplir el filtro.

Con ~11 000 hechos Spark es más lento que pandas (overhead de JVM). Se presenta porque el **modelo mental** —proyectar columnas, filtrar, agregar— es el mismo a otra escala. El recorte ya está listo para `spark.read.parquet(...)`.

Equivalente pandas de la consulta típica:

In [ ]:
resumen = (
    pd.read_parquet(
        ruta_hechos,
        columns=["comuna", "modalidad", "medio_transporte"],
        engine="pyarrow",
    )
    .groupby(["comuna", "modalidad"], observed=True)
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
resumen.head(10)

La misma consulta en Spark (no hace falta ejecutarla en este recorte):

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("hurtos").getOrCreate()
hechos_spk = spark.read.parquet("data/processed/hurtos_medellin_hechos.parquet")

(
    hechos_spk
    .select("comuna", "modalidad")
    .groupBy("comuna", "modalidad")
    .count()
    .orderBy("count", ascending=False)
    .show(10)
)
```

Criterio de cierre: existen `hurtos_medellin_hechos.parquet` y `hurtos_medellin_bienes.parquet` con tipos explícitos; el CSV procesado se conserva como entrega auditable; los conteos siguen haciéndose sobre **hechos**, no sobre bienes.